In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Predicting housing prices") \
    .config("spark.local.dir", "/tmp/spark-temp") \
    .getOrCreate()

In [5]:
data = spark.read.csv(r"C:\Users\Lincoln\Downloads\archive (1)\housing.csv", header=True, inferSchema=True)

data.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



In [6]:
clean_data = data.na.drop(subset=["total_bedrooms", "ocean_proximity"])

In [7]:
train_data, test_data = clean_data.randomSplit([0.8, 0.2], seed=42)

In [8]:
print("---  Re-Initialization Audit ---")
print(f"Raw Input Rows: {data.count()}")
print(f"Cleaned Data Rows: {clean_data.count()}")
print(f"Training Dataset Rows (80%): {train_data.count()}")
print(f"Testing Dataset Rows (20%): {test_data.count()}")

--- 🚦 Re-Initialization Audit ---
Raw Input Rows: 20640
Cleaned Data Rows: 20433
Training Dataset Rows (80%): 16395
Testing Dataset Rows (20%): 4038


In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator

indexer = StringIndexer(
    inputCol="ocean_proximity", 
    outputCol="ocean_proximity_index"
)

encoder = OneHotEncoder(
    inputCol="ocean_proximity_index", 
    outputCol="ocean_proximity_vec", 
    dropLast=False
)

feature_cols = [
    "housing_median_age", "total_rooms", "total_bedrooms", 
    "population", "households", "median_income", "ocean_proximity_vec"
]
assembler = VectorAssembler(
    inputCols=feature_cols, 
    outputCol="unscaled_features"
)

scaler = StandardScaler(
    inputCol="unscaled_features", 
    outputCol="features", 
    withMean=True, 
    withStd=True
)

    featuresCol="features", 
    labelCol="median_house_value", 
    regParam=0.001
)

pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, lr])

print("Executing automated production pipeline training... please wait...")
pipeline_model = pipeline.fit(train_data)
print("Pipeline training completely operational!")

test_predictions = pipeline_model.transform(test_data)

evaluator_mae = RegressionEvaluator(
    labelCol="median_house_value", 
    predictionCol="prediction", 
    metricName="mae"
)

final_pipeline_mae = evaluator_mae.evaluate(test_predictions)

print("\n--- Production Pipeline Evaluation ---")
print(f"Mean Absolute Error (MAE): ${final_pipeline_mae:,.2f}")

Executing automated production pipeline training... please wait...
Pipeline training completely operational!

--- 📊 Production Pipeline Evaluation ---
Mean Absolute Error (MAE): $50,597.34


In [10]:

evaluator_rmse = RegressionEvaluator(
    labelCol="median_house_value", 
    predictionCol="prediction", 
    metricName="rmse" # <-- Switching the target rule
)

final_pipeline_rmse = evaluator_rmse.evaluate(test_predictions)

print("---  Production Pipeline RMSE Audit ---")
print(f"Root Mean Squared Error (RMSE): ${final_pipeline_rmse:,.2f}")

--- 📊 Production Pipeline RMSE Audit ---
Root Mean Squared Error (RMSE): $69,825.72


In [11]:

evaluator_r2 = RegressionEvaluator(
    labelCol="median_house_value", 
    predictionCol="prediction", 
    metricName="r2" # <-- Tells Spark to calculate the variance percentage
)

final_pipeline_r2 = evaluator_r2.evaluate(test_predictions)

print("--- Complete Production Metrics Board ---")
print(f"1. Mean Absolute Error (MAE):       ${final_mae:,.2f}" if 'final_mae' in locals() else f"1. Mean Absolute Error (MAE):       $50,597.34")
print(f"2. Root Mean Squared Error (RMSE):  ${final_pipeline_rmse:,.2f}")
print(f"3. Variance Explained (R2 Score):  {final_pipeline_r2:.4f} ({final_pipeline_r2 * 100:.2f}% explained)")

--- 📊 Complete Production Metrics Board ---
1. Mean Absolute Error (MAE):       $50,597.34
2. Root Mean Squared Error (RMSE):  $69,825.72
3. Variance Explained (R2 Score):  0.6308 (63.08% explained)


In [13]:

model_directory = "california_housing_pipeline_model"

print(f"Saving your production model to '{model_directory}'...")
pipeline_model.save(model_directory)
print(" Model successfully saved to disk")

Saving your production model to 'california_housing_pipeline_model'...
💾 Model successfully saved to disk!


In [14]:
from pyspark.ml import PipelineModel

print("Loading the saved model from disk...")
loaded_pipeline_model = PipelineModel.load(model_directory)
print(" Model loaded completely intact")

production_predictions = loaded_pipeline_model.transform(test_data)

print("\n---  Live Loaded Model Predictions Verification ---")
production_predictions.select(
    "ocean_proximity", 
    "median_house_value", 
    "prediction"
).show(5, truncate=False)

Loading the saved model from disk...
🔄 Model loaded completely intact!

--- 🚀 Live Loaded Model Predictions Verification ---
+---------------+------------------+------------------+
|ocean_proximity|median_house_value|prediction        |
+---------------+------------------+------------------+
|NEAR OCEAN     |103600.0          |195064.51658992472|
|NEAR OCEAN     |106700.0          |234173.75777772983|
|NEAR OCEAN     |73200.0           |167248.4622126165 |
|NEAR OCEAN     |90100.0           |219152.42001227848|
|NEAR OCEAN     |67000.0           |174830.45942911576|
+---------------+------------------+------------------+
only showing top 5 rows


In [1]:
import os
print("Your notebook is physically located at:")
print(os.getcwd())

Your notebook is physically located at:
C:\Users\Lincoln
